Single End-to-End Agent

In [38]:
## Load Libraries
import os
import certifi
import requests
from dotenv import load_dotenv
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.agents import create_react_agent,AgentExecutor

In [35]:
## Load Environment Variables
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

OPEN_AI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [36]:
## Tavily Search Tool
search_tool = TavilySearchResults(max_results=2)
result=search_tool.invoke("How to make a cup of tea")


In [39]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [40]:
## Create a ChatOpenAI instance
llm=ChatOpenAI(model_name="gpt-4o-mini", temperature=0, openai_api_key=OPEN_AI_API_KEY)

result=llm.predict("Write a short poem about the beauty of nature.")

In [41]:
## Prompt

prompt=hub.pull("hwchase17/react")

c:\Users\OMKAR\Desktop\Agentic AI By Bappy\.venv\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [43]:
tools=[search_tool,get_weather_data]


In [45]:
agent=create_react_agent(llm=llm, tools=tools, prompt=prompt)

agent_executor=AgentExecutor(agent=agent, tools=tools, verbose=True)

response=agent_executor.invoke({"input":("Find the captial of India and then get the current weather in that city.")})

print(response["output"])



> Entering new AgentExecutor chain...
I need to find the capital of India first. The capital is a well-known fact, so I can answer that directly. After that, I will get the current weather for that city, which is New Delhi. 

Action: None needed for the capital, but I will proceed to get the weather.
Action Input: get_weather_data("New Delhi")None needed for the capital, but I will proceed to get the weather. is not a valid tool, try one of [tavily_search_results_json, get_weather_data].I need to find the capital of India first. The capital is a well-known fact, so I can answer that directly. After that, I will get the current weather for that city, which is New Delhi.

Action: get_weather_data
Action Input: "New Delhi"Could not fetch weather data for New DelhiIt seems there was an issue fetching the weather data for New Delhi. I will try to search for the current weather in New Delhi instead.

Action: tavily_search_results_json
Action Input: "current weather in New Delhi"[{'url': 'h